# 03 · A/B testing a retention action

The churn model tells us **who** is at risk. It does not tell us whether doing something about it **works**. Only an experiment answers that.

Scenario: the team wants to send a bonus to high-risk players on their first day. Before launching, we need to know (1) how many players the test needs, and (2) how to read the result.

The functions are in `src/churn/abtest.py`, written with plain formulas so every number can be checked by hand.

In [1]:
import json
import numpy as np
import pandas as pd

from churn.abtest import sample_size_per_group, two_proportion_ztest
from churn.data import load_players

df = load_players()
metrics = json.load(open("../reports/metrics.json"))
high_risk_churn = metrics["models"]["gradient_boosting"]["top_20pct"]["churn_rate_in_top"]
print(f"overall churn {df.churned.mean():.1%}, churn among the top-20% riskiest players {high_risk_churn:.1%}")

overall churn 22.3%, churn among the top-20% riskiest players 46.8%


## 1. Sample size: how many players before we start?

To detect an **absolute** drop in churn (e.g. 47% to 42% is a 5-point drop) with:
- **alpha = 0.05**: 5% chance of a false alarm,
- **power = 0.8**: 80% chance of catching the effect if it is real.

Smaller effects need many more players, and this has to be decided *before* the test, not after.

In [2]:
effects = [-0.02, -0.03, -0.05, -0.08, -0.10]
pd.DataFrame({
    "drop in churn (points)": [abs(e) * 100 for e in effects],
    "players per group": [sample_size_per_group(high_risk_churn, e) for e in effects],
}).set_index("drop in churn (points)")

,players per group
drop in churn (points),
2.0,9741
3.0,4321
5.0,1549
8.0,600
10.0,381


This dataset has about 1,800 high-risk players over four months (20% of ~9,000), so about 900 per group. That is enough for a 10-point effect, not even close for 2 points (9,741 per group, years of traffic). That is a real planning conclusion: either aim for a bigger effect, run the test longer, or test on all players.

## 2. Reading a result: the two-proportion z-test

Example on real data: is churn different on iOS and Android? (This is an *observational* comparison, not an experiment, so even a significant result would not prove the platform *causes* it. It is used here only to show how the test is read.)

In [3]:
g = df.groupby("operating_system").churned.agg(["sum", "count"])
r = two_proportion_ztest(g.loc["ANDROID", "sum"], g.loc["ANDROID", "count"],
                         g.loc["IOS", "sum"], g.loc["IOS", "count"])
print(f"Android {r.rate_a:.1%} vs iOS {r.rate_b:.1%}: difference {r.diff:+.1%}, "
      f"95% CI [{r.ci_low:+.1%}, {r.ci_high:+.1%}], p = {r.p_value:.2f}")

Android 22.7% vs iOS 21.8%: difference -0.9%, 95% CI [-2.6%, +0.8%], p = 0.31


- **p-value**: if there were truly no difference, how often would we see a gap at least this big just by chance? A large p-value means the gap is easily explained by noise.
- **95% confidence interval**: the range of differences consistent with the data. If it contains 0, we cannot rule out "no difference".

## 3. A simulated experiment

Simulated data (clearly not real): 2,000 high-risk players per group, control churns at the observed rate, and we *assume* the bonus lowers churn by 5 points.

In [4]:
rng = np.random.default_rng(7)
n = 2000
control = rng.binomial(n, high_risk_churn)
treatment = rng.binomial(n, high_risk_churn - 0.05)
r = two_proportion_ztest(control, n, treatment, n)
print(f"control {r.rate_a:.1%}, treatment {r.rate_b:.1%}, difference {r.diff:+.1%} "
      f"(95% CI {r.ci_low:+.1%} to {r.ci_high:+.1%}), p = {r.p_value:.4f}")

control 46.7%, treatment 40.6%, difference -6.1% (95% CI -9.1% to -3.0%), p = 0.0001


## 4. Why alpha matters: an A/A simulation

Run 2,000 fake tests where both groups have **the same** true churn rate. About 5% of them still come out "significant". That is what alpha = 0.05 means, and it is why peeking at a test every day and stopping at the first p < 0.05 inflates false alarms.

In [5]:
runs, n, p = 2000, 2000, high_risk_churn
false_alarms = sum(
    two_proportion_ztest(rng.binomial(n, p), n, rng.binomial(n, p), n).p_value < 0.05
    for _ in range(runs)
)
print(f"{false_alarms / runs:.1%} of A/A tests were 'significant' (expected about 5%)")

5.3% of A/A tests were 'significant' (expected about 5%)


## Takeaways

1. Decide the minimum effect worth detecting and the sample size **before** the test.
2. Report the **difference with its confidence interval**, not only the p-value.
3. Use the model to choose *who* to target, and the experiment to prove the action *works*.